Phase 2: Drug molecular feature preparation 

the main of this is to convert the molecular structures of PRISM drugs from the SMILES strings into the numeric molecular fingerprints. 

PRISM drug-response data
↓
Identify drugs with usable IC50 values
↓
Extract unique drug–SMILES pairs
↓
Validate molecular structures
↓
Generate Morgan fingerprints using RDKit
↓
Evaluate fingerprint coverage
↓
Save drug molecular features

load the libraries 

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries loaded successfully")

Libraries loaded successfully


load the PRISM Response Data 

In [3]:
response_path = "../data/raw/secondary-screen-dose-response-curve-parameters.csv"

response = pd.read_csv(response_path)

print("PRISM response shape:", response.shape)

PRISM response shape: (701004, 20)


C:\Users\SREEKUTTY\AppData\Local\Temp\ipykernel_34704\1530075993.py:3: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  response = pd.read_csv(response_path)


select the 28 GraphTCDR pancreatic cell lines

Phase 1 identified 28 pancreatic cell lines that are present in both:

- the PRISM dataset
- the GraphTCDR multi-omics dataset

We use the same 28 cell lines here so that the drug features and response data remain consistent with the multi-omics data.

In [4]:
cell_info = pd.read_csv(
    "../data/raw/secondary-screen-cell-line-info.csv"
)

graphtcdr_cells = pd.read_csv(
    "../data/raw/cellLineName402.csv"
)

pancreatic_cells = cell_info[
    cell_info["primary_tissue"].astype(str).str.lower() == "pancreas"
].copy()

pancreatic_names = set(
    pancreatic_cells["ccle_name"].astype(str).str.upper()
)

graphtcdr_pancreatic = graphtcdr_cells[
    graphtcdr_cells["cellLineName"]
    .astype(str)
    .str.upper()
    .isin(pancreatic_names)
].copy()

pancreatic_response_28 = response[
    response["depmap_id"].isin(
        pancreatic_cells[
            pancreatic_cells["ccle_name"].isin(
                graphtcdr_pancreatic["cellLineName"]
            )
        ]["depmap_id"]
    )
].copy()

print("Pancreatic GraphTCDR cell lines:", len(graphtcdr_pancreatic))
print("PRISM response records:", len(pancreatic_response_28))

Pancreatic GraphTCDR cell lines: 28
PRISM response records: 40119


keeping the only drugs with the usable 1c50 values 

## 3. Identify Drugs with Usable IC50 Values

For molecular feature preparation, we focus on drug–response records with finite IC50 values.

Missing or non-finite IC50 values are not treated as zero because a missing measurement does not represent zero drug response.

This gives us the drug set that can be directly linked to the response prediction task.

In [5]:
usable_response = pancreatic_response_28.copy()

usable_response["ic50"] = pd.to_numeric(
    usable_response["ic50"],
    errors="coerce"
)

usable_response = usable_response[
    np.isfinite(usable_response["ic50"])
].copy()

print("Usable IC50 records:", len(usable_response))
print("Unique drugs:", usable_response["name"].nunique())
print("Unique cell lines:", usable_response["depmap_id"].nunique())

Usable IC50 records: 18912
Unique drugs: 1228
Unique cell lines: 28


creation a unique drug-SMILES table 

A drug may appear many times in the PRISM response data because it is tested across multiple cell lines.

For molecular feature generation, we only need to process each drug structure once.

Therefore, we create a unique table containing:

- Drug name
- SMILES

This avoids generating the same molecular fingerprint repeatedly.

In [6]:
drug_table = (
    usable_response[["name", "smiles"]]
    .dropna(subset=["name", "smiles"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique drug structures:", len(drug_table))
print(drug_table.head())

Unique drug structures: 1228
          name                                             smiles
0  floxuridine  OC[C@H]1OC(C[C@H]1O)n1cc(F)c(=O)[nH]c1=O, OC[C...
1   valrubicin  CCCCC(=O)OCC(=O)[C@@]1(O)C[C@H](O[C@H]2C[C@H](...
2    adapalene  COc1ccc(cc1C12CC3CC(CC(C3)C1)C2)-c1ccc2cc(ccc2...
3  fulvestrant  C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)[C@H](C...
4   belinostat           ONC(=O)\C=C\c1cccc(c1)S(=O)(=O)Nc1ccccc1


checking the SMILES structure  format 

Before generating fingerprints, we check whether each drug has:

- one SMILES structure, or
- multiple comma-separated structures.

This is important because molecular fingerprint generation requires us to decide how to represent records containing multiple structures.

In [7]:
drug_table["smiles_count"] = (
    drug_table["smiles"]
    .astype(str)
    .str.split(",")
    .str.len()
)

print("Drugs with one SMILES:", (drug_table["smiles_count"] == 1).sum())
print("Drugs with multiple SMILES:", (drug_table["smiles_count"] > 1).sum())
print("Maximum SMILES per drug:", drug_table["smiles_count"].max())

Drugs with one SMILES: 559
Drugs with multiple SMILES: 669
Maximum SMILES per drug: 7


669/1,228 drugs have multiple comma-separated SMILES, so we must handle  that explicitly rather than blindly fingerprinting the whole string.

Inspect the multiple-SMILES records

Some PRISM drug entries contain more than one comma-separated SMILES.

Before generating molecular fingerprints, we inspect these entries to determine how they are represented and choose a consistent molecular-feature strategy.

In [8]:
multi_smiles = drug_table[
    drug_table["smiles_count"] > 1
].copy()

print("Number of multiple-SMILES drugs:", len(multi_smiles))

print("\nExamples:")
print(
    multi_smiles[["name", "smiles", "smiles_count"]]
    .head(10)
    .to_string(index=False)
)

Number of multiple-SMILES drugs: 669

Examples:
              name                                                                                                                                                                                                                                                                                                                                                                             smiles  smiles_count
       floxuridine                                                                                                                                                                                                                                                                                            OC[C@H]1OC(C[C@H]1O)n1cc(F)c(=O)[nH]c1=O, OC[C@H]1O[C@H](C[C@@H]1O)n1cc(F)c(=O)[nH]c1=O             2
         adapalene                                                                                                                              

Check whether the multiple SMILES are identical

Multiple SMILES in a PRISM record may simply repeat the same molecular structure.

We therefore check whether the comma-separated SMILES strings are identical or whether they represent different structures.

This determines how the drug should be represented by a single molecular fingerpr

In [9]:
def count_unique_smiles(smiles):
    return len(set(str(smiles).split(",")))

multi_smiles["unique_smiles_count"] = (
    multi_smiles["smiles"]
    .apply(count_unique_smiles)
)

print(
    "Multiple-SMILES drugs with identical SMILES only:",
    (multi_smiles["unique_smiles_count"] == 1).sum()
)

print(
    "Multiple-SMILES drugs with >1 unique SMILES:",
    (multi_smiles["unique_smiles_count"] > 1).sum()
)

Multiple-SMILES drugs with identical SMILES only: 0
Multiple-SMILES drugs with >1 unique SMILES: 669


Compare the molecular structures

Some drugs have multiple unique SMILES because a compound can have different
stereochemical or structural representations.

We inspect the number of unique molecular structures after converting the
SMILES strings into RDKit molecular objects.

This helps determine how to represent drugs with multiple structures
consistently.

In [11]:
from rdkit import Chem

def valid_molecule_count(smiles):
    molecules = []

    for s in str(smiles).split(","):
        mol = Chem.MolFromSmiles(s.strip())
        if mol is not None:
            molecules.append(mol)

    return len(molecules)

multi_smiles["valid_molecule_count"] = (
    multi_smiles["smiles"]
    .apply(valid_molecule_count)
)

print(
    "Multiple-SMILES drugs:",
    len(multi_smiles)
)

print(
    "Drugs with all SMILES valid:",
    (
        multi_smiles["valid_molecule_count"]
        == multi_smiles["unique_smiles_count"]
    ).sum()
)

print(
    "Drugs with at least one invalid SMILES:",
    (
        multi_smiles["valid_molecule_count"]
        < multi_smiles["unique_smiles_count"]
    ).sum()
)

Multiple-SMILES drugs: 669
Drugs with all SMILES valid: 395
Drugs with at least one invalid SMILES: 0


Check whether the multiple SMILES differ only by stereochemistry

Some drugs contain multiple valid SMILES because different molecular
representations may encode stereochemical variants.

We compare the molecules after removing stereochemical information.

If multiple SMILES collapse to the same non-stereochemical structure, the
difference is primarily stereochemical representation.

In [12]:
def count_unique_nonstereo(smiles):
    nonstereo = []

    for s in str(smiles).split(","):
        mol = Chem.MolFromSmiles(s.strip())

        if mol is not None:
            mol = Chem.Mol(mol)
            Chem.RemoveStereochemistry(mol)
            nonstereo.append(
                Chem.MolToSmiles(mol, canonical=True)
            )

    return len(set(nonstereo))

multi_smiles["unique_nonstereo_count"] = (
    multi_smiles["smiles"]
    .apply(count_unique_nonstereo)
)

print(
    "Multiple-SMILES drugs:",
    len(multi_smiles)
)

print(
    "Same structure without stereochemistry:",
    (
        multi_smiles["unique_nonstereo_count"] == 1
    ).sum()
)

print(
    "Different structures even without stereochemistry:",
    (
        multi_smiles["unique_nonstereo_count"] > 1
    ).sum()
)

Multiple-SMILES drugs: 669
Same structure without stereochemistry: 663
Different structures even without stereochemistry: 6


Generate a canonical SMILES for each drug

Create a Standard Molecular Representation

To generate molecular fingerprints consistently, each drug needs one standardized
molecular representation.

For drugs with multiple SMILES, the first valid molecular structure is used
as the representative structure. The resulting molecule is converted back to
a canonical SMILES using RDKit.

This creates one standardized structure per PRISM drug.

In [13]:
def get_canonical_smiles(smiles):
    for s in str(smiles).split(","):
        mol = Chem.MolFromSmiles(s.strip())

        if mol is not None:
            return Chem.MolToSmiles(
                mol,
                canonical=True
            )

    return None


drug_table["canonical_smiles"] = (
    drug_table["smiles"]
    .apply(get_canonical_smiles)
)

print(
    "Total drugs:",
    len(drug_table)
)

print(
    "Valid canonical SMILES:",
    drug_table["canonical_smiles"].notna().sum()
)

print(
    "Missing canonical SMILES:",
    drug_table["canonical_smiles"].isna().sum()
)

Total drugs: 1228
Valid canonical SMILES: 1228
Missing canonical SMILES: 0


Generate Morgan fingerprints

The canonical SMILES are converted into Morgan fingerprints using RDKit.

Morgan fingerprints represent the local chemical structure of a molecule as a
fixed-length binary vector.

We use:

- Radius = 2
- Fingerprint size = 2048 bits

These fingerprints will serve as the numerical drug features for the
subsequent drug-response prediction models.

In [14]:
from rdkit.Chem import AllChem

def generate_morgan_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=2048
    )

drug_table["morgan_fp"] = (
    drug_table["canonical_smiles"]
    .apply(generate_morgan_fingerprint)
)

print("Drugs processed:", len(drug_table))
print("Fingerprint size:", len(drug_table["morgan_fp"].iloc[0]))

[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerat

Drugs processed: 1228
Fingerprint size: 2048


[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerator
[15:22:38] DEPRECATION WARNING: please use MorganGenerat

Convert fingerprints into a numerical matrix

The RDKit Morgan fingerprints are currently stored as RDKit fingerprint
objects.

We convert them into a numerical binary matrix where:

- Each row represents one drug.
- Each column represents one fingerprint bit.
- Each drug is represented by 2048 molecular features.

This matrix will be used as the drug-feature input for machine-learning models

In [15]:
from rdkit import DataStructs

fingerprint_matrix = np.zeros(
    (len(drug_table), 2048),
    dtype=np.uint8
)

for i, fp in enumerate(drug_table["morgan_fp"]):
    DataStructs.ConvertToNumpyArray(
        fp,
        fingerprint_matrix[i]
    )

print("Drug feature matrix shape:", fingerprint_matrix.shape)
print("Data type:", fingerprint_matrix.dtype)

Drug feature matrix shape: (1228, 2048)
Data type: uint8


Check fingerprint quality

Validate the Drug Fingerprints

We check the generated fingerprint matrix to make sure:

- All 1,228 drugs have fingerprints.
- Every fingerprint contains 2,048 bits.
- The fingerprints contain both active (1) and inactive (0) bits.

This confirms that the molecular feature matrix is suitable for downstream modeling.

In [16]:
print("Number of drugs:", fingerprint_matrix.shape[0])
print("Number of fingerprint features:", fingerprint_matrix.shape[1])
print("Minimum bit value:", fingerprint_matrix.min())
print("Maximum bit value:", fingerprint_matrix.max())
print("Total active bits:", fingerprint_matrix.sum())

Number of drugs: 1228
Number of fingerprint features: 2048
Minimum bit value: 0
Maximum bit value: 1
Total active bits: 60108


Save Drug Molecular Features


The validated molecular fingerprints are saved to the processed-data directory.

This allows the same drug features to be reused in later modeling stages
without regenerating the fingerprints.

In [17]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

np.save(
    processed_dir / "drug_morgan_fingerprints_1228x2048.npy",
    fingerprint_matrix
)

drug_table[
    ["name", "canonical_smiles"]
].to_csv(
    processed_dir / "drug_smiles_1228.csv",
    index=False
)

print("Saved successfully.")
print("Fingerprint file:", processed_dir / "drug_morgan_fingerprints_1228x2048.npy")
print("Drug metadata file:", processed_dir / "drug_smiles_1228.csv")

Saved successfully.
Fingerprint file: ..\data\processed\drug_morgan_fingerprints_1228x2048.npy
Drug metadata file: ..\data\processed\drug_smiles_1228.csv
